# NB13 – Baseline Detector Experiments (v3 — Two-Phase)

## Why previous versions OOM'd
Every previous version streamed 22 GB of parquet shards through the DataLoader.
Building 3 dataset objects × 70K row maps + HF cache files exceeded Kaggle's 13 GB system RAM.

## This version — two-phase architecture
**Phase 1 (once, ~11 min):** Download shards one at a time, decode PNG → resize 224×224 → re-encode JPEG 85%. Write to a single `/kaggle/working/compact.h5`. Result: **~630 MB** for all 70K images. Push to HF every 10 min so it survives restarts.

**Phase 2 (training):** Load ONLY from local `compact.h5`. Zero HF calls during training. Standard DataLoader with `num_workers=4`.

## Resume on restart
- Cell 4 downloads `compact.h5` from HF and continues extraction from where it stopped
- Cell 8 downloads the best checkpoint from HF and resumes training from the next epoch
- SIGTERM / Ctrl+C triggers immediate push before exit

In [1]:
# ── 0. Install deps ───────────────────────────────────────────────────────────
import importlib, subprocess, sys
def _pip(*pkgs): subprocess.run([sys.executable,'-m','pip','install','-q',*pkgs],check=True)
need = []
for mod,pkg in [('huggingface_hub','huggingface_hub>=0.23'),('pyarrow','pyarrow'),
                ('PIL','pillow'),('sklearn','scikit-learn'),('timm','timm>=0.9'),('h5py','h5py')]:
    if importlib.util.find_spec(mod) is None: need.append(pkg)
if need: _pip(*need); print('installed:',need)
else: print('all deps present')
import torch
print(f'torch {torch.__version__} | cuda={torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p=torch.cuda.get_device_properties(i)
        print(f'  GPU{i}: {p.name}  {p.total_memory/1e9:.1f} GB')

all deps present
torch 2.10.0+cu128 | cuda=True
  GPU0: Tesla T4  15.6 GB
  GPU1: Tesla T4  15.6 GB


In [ ]:
# ── 1. Config ─────────────────────────────────────────────────────────────────
import os,gc,io,json,time,random,signal,threading,tempfile
from pathlib import Path
import numpy as np

# CUDA memory fragmentation fix — must be set before torch import
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

REPO_ID        = 'Shanmuk4622/ai-detection-dataset-v2'
RESULTS_PREFIX = 'results/baseline_experiments'

RUN_RESNET50     = True
RUN_EFFICIENTNET = True
RUN_VIT          = True

EPOCHS        = 10
BATCH_SIZE    = 64      # reduced from 128 — prevents OOM on EfficientNet/ViT with DataParallel
LR            = 3e-4
IMG_SIZE      = 224
NUM_WORKERS   = 2       # reduced from 4 — persistent workers hold pinned memory
MASTER_SEED   = 42
PUSH_INTERVAL = 1200    # push to HF max every 20 min

WORKING      = Path('/kaggle/working')
COMPACT_H5   = WORKING / 'compact.h5'
COMPACT_REPO = f'{RESULTS_PREFIX}/compact.h5'
CKPT_DIR     = WORKING / 'checkpoints'
HF_CACHE     = Path('/kaggle/temp/hf_cache')
for d in [CKPT_DIR, HF_CACHE]: d.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE)

import torch
random.seed(MASTER_SEED); np.random.seed(MASTER_SEED); torch.manual_seed(MASTER_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(MASTER_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
free_gb = os.statvfs('/kaggle/working').f_bavail * os.statvfs('/kaggle/working').f_frsize / 1e9
print(f'Config OK | device={DEVICE} | /kaggle/working free: {free_gb:.1f} GB')
print(f'BATCH_SIZE={BATCH_SIZE} | NUM_WORKERS={NUM_WORKERS} | PYTORCH_ALLOC_CONF=expandable_segments:True')


In [3]:
# ── 2. HF auth + common library ───────────────────────────────────────────────
from huggingface_hub import HfApi, hf_hub_download, CommitOperationAdd, list_repo_files
from huggingface_hub.utils import HfHubHTTPError

def load_token():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret('HF_TOKEN')
        if t: return t.strip()
    except Exception: pass
    for k in ('HF_TOKEN','HUGGINGFACE_TOKEN'):
        v=os.environ.get(k)
        if v: return v.strip()
    import getpass; return getpass.getpass('HF write token: ').strip()

TOKEN = load_token()
os.environ['HF_TOKEN'] = TOKEN
api = HfApi(token=TOKEN)
lib = hf_hub_download(REPO_ID,'ai_detect_common.py',repo_type='dataset',token=TOKEN)
import importlib.util
spec=importlib.util.spec_from_file_location('ai_detect_common',lib)
C=importlib.util.module_from_spec(spec); spec.loader.exec_module(C)
cfg=C.read_config(REPO_ID,TOKEN)
GENERATORS=list(cfg['generators'].keys())
print(f'Library {C.PIPELINE_VERSION} | generators: {GENERATORS}')

ai_detect_common.py: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Library 1.2 | generators: ['sd15', 'sdxl', 'flux_schnell', 'kandinsky22', 'pixart_sigma', 'sd_cascade']


In [4]:
# ── 3. Push helpers (rate-limit safe + SIGTERM) ───────────────────────────────
_last_push=[time.time()]; _push_lock=threading.Lock(); _exit_flag=[False]
_commit_count=[0]; _hour_start=[time.time()]
HF_RATE_LIMIT=100

def _rate_guard():
    now=time.time()
    if now-_hour_start[0]>=3600: _commit_count[0]=0; _hour_start[0]=now
    if _commit_count[0]>=HF_RATE_LIMIT:
        wait=3600-(now-_hour_start[0])+10
        print(f'  [RATE LIMIT] {_commit_count[0]} commits — sleeping {wait:.0f}s')
        time.sleep(wait); _commit_count[0]=0; _hour_start[0]=time.time()

def robust_commit(operations,msg,retries=6):
    _rate_guard()
    delay=5.0
    for attempt in range(1,retries+1):
        try:
            api.create_commit(repo_id=REPO_ID,repo_type='dataset',operations=operations,commit_message=msg)
            _commit_count[0]+=1; return True
        except HfHubHTTPError as e:
            code=getattr(getattr(e,'response',None),'status_code',None)
            if code==429: print('  [429] sleep 60s'); time.sleep(60); continue
            if attempt==retries or code not in (500,502,503,504,None): raise
            print(f'  HF retry {attempt}/{retries} HTTP{code}; sleep {delay:.0f}s')
            time.sleep(delay); delay=min(delay*2,300)
        except Exception as e:
            if attempt==retries: raise
            time.sleep(delay); delay=min(delay*2,300)

def push_file(local_path,repo_path,msg):
    with _push_lock:
        op=CommitOperationAdd(path_in_repo=repo_path,path_or_fileobj=str(local_path))
        robust_commit([op],msg); _last_push[0]=time.time()
        print(f'  ↑ HF: {repo_path}')

def push_json(data,repo_path,msg):
    tmp=Path(tempfile.mktemp(suffix='.json'))
    tmp.write_text(json.dumps(data,indent=2))
    push_file(tmp,repo_path,msg); tmp.unlink(missing_ok=True)

def should_push(): return (time.time()-_last_push[0])>=PUSH_INTERVAL

def _on_sigterm(sig=None,frame=None):
    _exit_flag[0]=True
    print('\n[SIGTERM] will push immediately on next checkpoint')
signal.signal(signal.SIGTERM,_on_sigterm)
print(f'Push helpers ready | rate limit: {HF_RATE_LIMIT}/hr | interval: {PUSH_INTERVAL//60} min')

Push helpers ready | rate limit: 100/hr | interval: 20 min


In [5]:
# ── 4. Phase 1 — Build compact.h5 (one-time, fully resumable) ────────────────
#
# ROOT CAUSE FIX:
#   h5py.special_dtype(vlen=bytes) creates a VLEN STRING dataset.
#   VLEN strings reject embedded null bytes (0x00), and JPEG data is full of them.
#   FIX: use h5py.special_dtype(vlen=np.uint8) — a true variable-length BINARY
#   dataset that stores numpy uint8 arrays. Null bytes are fine.
#
# Also note: manifest reports 888 shards (not 248 estimated). That is fine —
# we delete each shard from /kaggle/temp after processing, so disk never fills up.

import h5py, pyarrow.parquet as pq
from PIL import Image

EXTRACT_PROGRESS_REPO = f'{RESULTS_PREFIX}/nb13_extract_progress.json'
COMPACT_PUSH_INTERVAL = 600

# Correct dtype: vlen=np.uint8 stores variable-length byte arrays, not strings
VLEN_UINT8 = h5py.special_dtype(vlen=np.uint8)

def _jpeg_encode(png_bytes, size=224):
    img = Image.open(io.BytesIO(png_bytes)).convert('RGB')
    img = img.resize((size, size), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=85, subsampling=0)
    return buf.getvalue()

def _parquet_ok(path):
    try:
        with open(path, 'rb') as f:
            h = f.read(4); f.seek(-4, 2); t = f.read(4)
        return h == b'PAR1' and t == b'PAR1'
    except:
        return False

def build_compact_h5():
    import shutil

    # Within-session resume
    if COMPACT_H5.exists():
        with h5py.File(COMPACT_H5, 'r') as f:
            n_done = len(f['image_id'])
        if n_done >= 70000:
            print(f'compact.h5 complete ({n_done} images). Skipping Phase 1.')
            return
        print(f'compact.h5 exists locally: {n_done} images — resuming')
    else:
        # Cross-session resume: download from HF
        repo_files = list(list_repo_files(REPO_ID, repo_type='dataset', token=TOKEN))
        if COMPACT_REPO in repo_files:
            print('Downloading compact.h5 from HF for resume...')
            dl = hf_hub_download(REPO_ID, COMPACT_REPO, repo_type='dataset',
                                  token=TOKEN, local_dir=str(WORKING))
            dst = COMPACT_H5
            if str(Path(dl).resolve()) != str(dst.resolve()):
                shutil.copy2(dl, dst)
            with h5py.File(COMPACT_H5, 'r') as f:
                n_done = len(f['image_id'])
            print(f'Resumed from HF: {n_done} images already done')
        else:
            print('No existing compact.h5 — starting fresh')
            n_done = 0

    # Load manifest
    man_path = hf_hub_download(REPO_ID, 'manifest.parquet', repo_type='dataset', token=TOKEN)
    manifest = pq.read_table(man_path).to_pydict()
    total    = len(manifest['image_id'])
    split_map = {iid: sp for iid, sp in zip(manifest['image_id'], manifest['split'])}
    print(f'Manifest: {total} images')

    # Build set of already-extracted image_ids
    done_ids = set()
    if COMPACT_H5.exists() and n_done > 0:
        with h5py.File(COMPACT_H5, 'r') as f:
            done_ids = set(x.decode() for x in f['image_id'][:])
        print(f'Already done: {len(done_ids)}')

    # List all image shards
    repo_files = list(list_repo_files(REPO_ID, repo_type='dataset', token=TOKEN))
    shard_files = sorted(
        sf for sf in repo_files
        if sf.endswith('.parquet') and ('real/' in sf or 'data/' in sf)
    )
    print(f'Total shards: {len(shard_files)}')

    last_h5_push = time.time()
    total_written = n_done

    with h5py.File(COMPACT_H5, 'a') as h5:
        if 'image_id' not in h5:
            h5.create_dataset('image_id',   shape=(0,), maxshape=(None,),
                              dtype=h5py.string_dtype(), chunks=True)
            h5.create_dataset('label',      shape=(0,), maxshape=(None,),
                              dtype='int8', chunks=True)
            h5.create_dataset('generator',  shape=(0,), maxshape=(None,),
                              dtype=h5py.string_dtype(), chunks=True)
            h5.create_dataset('split',      shape=(0,), maxshape=(None,),
                              dtype=h5py.string_dtype(), chunks=True)
            # CORRECT binary vlen — stores numpy uint8 arrays, handles null bytes
            h5.create_dataset('image_jpeg', shape=(0,), maxshape=(None,),
                              dtype=VLEN_UINT8, chunks=True)
            print('HDF5 created — image_jpeg dtype: vlen uint8 (binary-safe)')

        for sidx, shard_path in enumerate(shard_files):
            local = hf_hub_download(
                REPO_ID, shard_path, repo_type='dataset',
                token=TOKEN, local_dir=str(HF_CACHE)
            )

            if not _parquet_ok(local):
                print(f'  WARN: corrupted {shard_path} — re-downloading')
                Path(local).unlink(missing_ok=True)
                local = hf_hub_download(
                    REPO_ID, shard_path, repo_type='dataset',
                    token=TOKEN, local_dir=str(HF_CACHE), force_download=True
                )
                if not _parquet_ok(local):
                    print(f'  ERROR: still corrupted — skipping {shard_path}')
                    continue

            try:
                tbl = pq.read_table(
                    local, columns=['image_id','image','label','generator']
                ).to_pydict()
            except Exception as e:
                print(f'  ERROR reading {shard_path}: {e} — skipping')
                try: Path(local).unlink()
                except: pass
                continue

            new_ids, new_jpegs, new_labels, new_gens, new_splits = [], [], [], [], []
            for iid, png, lbl, gen in zip(
                tbl['image_id'], tbl['image'], tbl['label'], tbl['generator']
            ):
                if iid in done_ids:
                    continue
                try:
                    jpeg_bytes = _jpeg_encode(png)
                except Exception as e:
                    print(f'  WARN encode {iid}: {e}')
                    continue
                new_ids.append(iid)
                # np.frombuffer gives uint8 array — matches VLEN_UINT8 dtype
                new_jpegs.append(np.frombuffer(jpeg_bytes, dtype=np.uint8))
                new_labels.append(lbl)
                new_gens.append(gen)
                new_splits.append(split_map.get(iid, 'train'))
                done_ids.add(iid)

            if new_ids:
                n_cur = len(h5['image_id'])
                n_new = len(new_ids)
                h5['image_id'].resize(n_cur + n_new, axis=0)
                h5['image_id'][n_cur:] = [s.encode() for s in new_ids]
                h5['label'].resize(n_cur + n_new, axis=0)
                h5['label'][n_cur:] = new_labels
                h5['generator'].resize(n_cur + n_new, axis=0)
                h5['generator'][n_cur:] = [s.encode() for s in new_gens]
                h5['split'].resize(n_cur + n_new, axis=0)
                h5['split'][n_cur:] = [s.encode() for s in new_splits]
                h5['image_jpeg'].resize(n_cur + n_new, axis=0)
                for k, arr in enumerate(new_jpegs):
                    h5['image_jpeg'][n_cur + k] = arr  # write uint8 array directly
                h5.flush()
                total_written += n_new

            if (sidx + 1) % 5 == 0 or sidx == 0:
                mb = COMPACT_H5.stat().st_size / 1e6
                pct = total_written / total * 100
                print(f'  shard {sidx+1}/{len(shard_files)} | '
                      f'{total_written}/{total} ({pct:.1f}%) | {mb:.0f} MB')

            # Push to HF on schedule
            if (time.time() - last_h5_push) >= COMPACT_PUSH_INTERVAL:
                h5.flush()
                push_file(COMPACT_H5, COMPACT_REPO,
                          f'Phase1: {total_written}/{total} images')
                last_h5_push = time.time()

            # Delete shard from /kaggle/temp to keep disk clean
            try: Path(local).unlink()
            except: pass

        final_count = len(h5['image_id'])

    mb = COMPACT_H5.stat().st_size / 1e6
    print(f'Extraction complete: {final_count} images | {mb:.0f} MB')
    push_file(COMPACT_H5, COMPACT_REPO, f'Phase1 COMPLETE: {final_count} images')
    push_json({'total': final_count, 'completed_utc': C.now_utc()},
               EXTRACT_PROGRESS_REPO, 'Phase1 complete')
    print('compact.h5 pushed to HF.')


build_compact_h5()
print('Phase 1 done.')


results/baseline_experiments/compact.h5:   0%|          | 0.00/722M [00:00<?, ?B/s]

Resumed from HF: 45357 images already done


manifest.parquet:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

Manifest: 70000 images
Already done: 45357
Total shards: 888


data/flux_schnell/flux_schnell-2a42f202-(…):   0%|          | 0.00/10.4M [00:00<?, ?B/s]

...

real/real-70beac4d-00019.parquet:   0%|          | 0.00/203M [00:00<?, ?B/s]

Extraction complete: 70000 images | 1058 MB


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  ↑ HF: results/baseline_experiments/compact.h5
  ↑ HF: results/baseline_experiments/nb13_extract_progress.json
compact.h5 pushed to HF.
Phase 1 done.


In [6]:
# ── 5. Verify compact.h5 ─────────────────────────────────────────────────────
import h5py
from collections import Counter
from PIL import Image

with h5py.File(COMPACT_H5,'r') as f:
    n=len(f['image_id']); labels=f['label'][:]
    gens=[x.decode() for x in f['generator'][:]]
    splits=[x.decode() for x in f['split'][:]]
    img=Image.open(io.BytesIO(bytes(f['image_jpeg'][0])))

print(f'compact.h5: {n} images | {COMPACT_H5.stat().st_size/1e6:.0f} MB')
print(f'Labels: real={sum(1 for l in labels if l==0)} AI={sum(1 for l in labels if l==1)}')
print(f'Generators: {dict(Counter(gens))}')
print(f'Splits: {dict(Counter(splits))}')
print(f'Image: {img.size} mode={img.mode}')
assert n>=69000,f'Too few: {n}'
assert img.size==(224,224),f'Wrong size: {img.size}'
print('Verification PASSED')

compact.h5: 70000 images | 1058 MB
Labels: real=10000 AI=60000
Generators: {'flux_schnell': 10000, 'kandinsky22': 10000, 'pixart_sigma': 10000, 'sd15': 10000, 'sd_cascade': 10000, 'sdxl': 10000, 'real': 10000}
Splits: {'train': 49392, 'test': 10486, 'val': 10122}
Image: (224, 224) mode=RGB
Verification PASSED


In [ ]:
# ── 6. PyTorch Dataset — reads local HDF5 only (zero HF calls during training)
import h5py
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

TRAIN_TF=T.Compose([T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.1,contrast=0.1,saturation=0.1),
    T.ToTensor(),T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
EVAL_TF=T.Compose([T.ToTensor(),T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

class CompactDataset(Dataset):
    def __init__(self,h5_path,indices,transform):
        self.h5_path=str(h5_path); self.indices=indices; self.transform=transform; self._h5=None
        with h5py.File(h5_path,'r') as f:
            all_labels=f['label'][:]; all_gens=[x.decode() for x in f['generator'][:]]
        self.labels=[int(all_labels[i]) for i in indices]
        self.generators=[all_gens[i] for i in indices]
    def _open(self):
        if self._h5 is None: self._h5=h5py.File(self.h5_path,'r')
    def __len__(self): return len(self.indices)
    def __getitem__(self,idx):
        self._open()
        row=self.indices[idx]
        # vlen uint8 array → .tobytes() → bytes for PIL
        img=Image.open(io.BytesIO(self._h5['image_jpeg'][row].tobytes())).convert('RGB')
        return self.transform(img),self.labels[idx],self.generators[idx]
    def __del__(self):
        if self._h5 is not None:
            try: self._h5.close()
            except: pass

with h5py.File(COMPACT_H5,'r') as f:
    all_splits=[x.decode() for x in f['split'][:]]

split_idx={'train':[],'val':[],'test':[]}
for i,sp in enumerate(all_splits):
    if sp in split_idx: split_idx[sp].append(i)
print(f"Splits: train={len(split_idx['train'])} val={len(split_idx['val'])} test={len(split_idx['test'])}")

def make_loaders(train_idx=None,test_idx=None):
    tr=train_idx if train_idx is not None else split_idx['train']
    te=test_idx  if test_idx  is not None else split_idx['test']
    train_ds=CompactDataset(COMPACT_H5,tr,TRAIN_TF)
    val_ds  =CompactDataset(COMPACT_H5,split_idx['val'],EVAL_TF)
    test_ds =CompactDataset(COMPACT_H5,te,EVAL_TF)
    # persistent_workers=False — workers are torn down between epochs,
    # freeing all pinned memory before the next model loads
    kw=dict(num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=False)
    return (DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,**kw),
            DataLoader(val_ds,  batch_size=BATCH_SIZE,shuffle=False,**kw),
            DataLoader(test_ds, batch_size=BATCH_SIZE,shuffle=False,**kw))

def destroy_loaders(*loaders):
    """Explicitly delete DataLoaders and their datasets to free pinned memory."""
    for dl in loaders:
        if dl is not None:
            try:
                dl._iterator = None   # kill any live iterator
                del dl.dataset
                del dl
            except: pass
    gc.collect()

train_dl,val_dl,test_dl=make_loaders()
imgs,labels,gens=next(iter(train_dl))
print(f'Batch OK: {imgs.shape} | labels {labels[:8].tolist()}')


In [ ]:
# ── 7. Model factory ──────────────────────────────────────────────────────────
import timm
MODEL_SPECS={
    'resnet50':{'timm_name':'resnet50.a1_in1k','run':RUN_RESNET50,'description':'ResNet-50 (IN-1k)'},
    'efficientnet_b4':{'timm_name':'efficientnet_b4.ra2_in1k','run':RUN_EFFICIENTNET,'description':'EfficientNet-B4 (IN-1k)'},
    'vit_b16':{'timm_name':'vit_base_patch16_224.augreg_in21k_ft_in1k','run':RUN_VIT,'description':'ViT-B/16 (IN-21k→IN-1k)'},
}

def build_model(key):
    m=timm.create_model(MODEL_SPECS[key]['timm_name'],pretrained=True,num_classes=2)
    if torch.cuda.device_count()>1:
        m=torch.nn.DataParallel(m)
        print(f'  DataParallel × {torch.cuda.device_count()}')
    return m.to(DEVICE)

def free_model(model, optimizer=None, scheduler=None):
    """Thoroughly free GPU memory before loading next model."""
    # Zero all gradients first
    if optimizer is not None:
        optimizer.zero_grad(set_to_none=True)
        del optimizer
    if scheduler is not None:
        del scheduler
    # Move model to CPU first — flushes GPU allocations
    try:
        m = model.module if isinstance(model, torch.nn.DataParallel) else model
        m.cpu()
    except: pass
    if isinstance(model, torch.nn.DataParallel):
        del model.module
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    allocated = torch.cuda.memory_allocated()/1e6 if torch.cuda.is_available() else 0
    print(f'  GPU freed | allocated: {allocated:.0f} MB')

print('Models:',[k for k,v in MODEL_SPECS.items() if v['run']])


In [9]:
# ── 8. Checkpoint helpers ─────────────────────────────────────────────────────
def ckpt_local(k): return CKPT_DIR/f'{k}_best.pt'
def ckpt_repo(k):  return f'{RESULTS_PREFIX}/checkpoints/{k}_best.pt'
def prog_repo(k):  return f'{RESULTS_PREFIX}/{k}_progress.json'

def _unwrap(model):
    m=model.module if isinstance(model,torch.nn.DataParallel) else model
    return {k.replace('module.',''):v for k,v in m.state_dict().items()}

def load_checkpoint(model,optimizer,scheduler,key):
    try:
        files=list(list_repo_files(REPO_ID,repo_type='dataset',token=TOKEN))
        if ckpt_repo(key) in files:
            dl=hf_hub_download(REPO_ID,ckpt_repo(key),repo_type='dataset',token=TOKEN)
            ckpt=torch.load(dl,map_location=DEVICE,weights_only=False)
            m=model.module if isinstance(model,torch.nn.DataParallel) else model
            m.load_state_dict({k.replace('module.',''):v for k,v in ckpt['model_state'].items()},strict=False)
            optimizer.load_state_dict(ckpt['optimizer_state'])
            if scheduler and ckpt.get('scheduler_state'): scheduler.load_state_dict(ckpt['scheduler_state'])
            ep=ckpt['epoch']+1; auc=ckpt.get('best_val_auc',0.0)
            print(f'  resumed {key} from HF | ep={ckpt["epoch"]} auc={auc:.4f}'); return ep,auc
    except Exception as e: print(f'  no ckpt for {key} ({e.__class__.__name__}) — fresh start')
    return 0,0.0

def save_checkpoint(model,optimizer,scheduler,key,epoch,best_auc,history,force=False):
    local=ckpt_local(key)
    torch.save({'epoch':epoch,'model_state':_unwrap(model),'optimizer_state':optimizer.state_dict(),
                'scheduler_state':scheduler.state_dict() if scheduler else None,
                'best_val_auc':best_auc,'history':history,'saved_utc':C.now_utc()},local)
    if force or should_push() or _exit_flag[0]:
        push_file(local,ckpt_repo(key),f'{key}: ep{epoch} auc={best_auc:.4f}')
        push_json({'model':key,'history':history,'best_val_auc':best_auc,
                   'last_epoch':epoch,'saved_utc':C.now_utc()},prog_repo(key),f'{key}: ep{epoch}')
print('Checkpoint helpers ready')

Checkpoint helpers ready


In [ ]:
# ── 9. Train / eval loops ─────────────────────────────────────────────────────
import torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import roc_auc_score,f1_score,accuracy_score

def compute_metrics(al,ap,ad,ag):
    al,ap,ad,ag=map(np.array,[al,ap,ad,ag])
    res={'overall':{'auroc':float(roc_auc_score(al,ap)),'f1':float(f1_score(al,ad,zero_division=0)),
                   'accuracy':float(accuracy_score(al,ad)),'n':int(len(al))}}
    for gen in GENERATORS:
        mask=(ag==gen)|(ag=='real')
        if mask.sum()<10: continue
        gl,gp,gd=al[mask],ap[mask],ad[mask]
        try: auroc=float(roc_auc_score(gl,gp))
        except: auroc=None
        res[gen]={'auroc':auroc,'f1':float(f1_score(gl,gd,zero_division=0)),
                  'accuracy':float(accuracy_score(gl,gd)),'n':int(mask.sum())}
    return res

def train_epoch(model,loader,optimizer,criterion,epoch):
    model.train(); ls,cor,tot=0.0,0,0
    for step,(imgs,labels,_) in enumerate(loader):
        imgs,labels=imgs.to(DEVICE),labels.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)   # set_to_none saves memory vs zero_grad()
        logits=model(imgs); loss=criterion(logits,labels)
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
        preds=logits.argmax(1); ls+=loss.item()*labels.size(0)
        cor+=(preds==labels).sum().item(); tot+=labels.size(0)
        if (step+1)%50==0: print(f'    ep{epoch} step{step+1}/{len(loader)} loss={ls/tot:.4f} acc={cor/tot:.4f}')
    return {'loss':ls/tot,'accuracy':cor/tot}

@torch.no_grad()
def evaluate(model,loader):
    model.eval(); al,ap,ad,ag=[],[],[],[]
    for imgs,labels,gens in loader:
        logits=model(imgs.to(DEVICE)); probs=F.softmax(logits,1)[:,1]; preds=logits.argmax(1)
        al.extend(labels.tolist()); ap.extend(probs.cpu().tolist())
        ad.extend(preds.cpu().tolist()); ag.extend(list(gens))
    return compute_metrics(al,ap,ad,ag)

def train_model(key,tr_dl,v_dl):
    print(f'\n{"="*60}\nTRAINING: {key} — {MODEL_SPECS[key]["description"]}\n{"="*60}')
    model=build_model(key)
    criterion=nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS,eta_min=LR*0.01)
    start_ep,best_auc=load_checkpoint(model,optimizer,scheduler,key)
    if start_ep>=EPOCHS:
        print(f'  already {EPOCHS} epochs done — skipping to test eval')
        return model,optimizer,scheduler,[]
    history=[]
    for epoch in range(start_ep,EPOCHS):
        t0=time.time()
        tr=train_epoch(model,tr_dl,optimizer,criterion,epoch)
        val=evaluate(model,v_dl); scheduler.step()
        val_auc=val['overall']['auroc']; is_best=val_auc>best_auc
        if is_best: best_auc=val_auc
        elapsed=time.time()-t0
        rec={'epoch':epoch,'train_loss':tr['loss'],'train_acc':tr['accuracy'],
             'val_auc':val_auc,'val_acc':val['overall']['accuracy'],
             'val_per_gen':val,'elapsed_s':elapsed,'ts':C.now_utc()}
        history.append(rec)
        print(f'Ep {epoch:02d}/{EPOCHS-1} | loss={tr["loss"]:.4f} acc={tr["accuracy"]:.4f} | '
              f'val_auc={val_auc:.4f} {"★ BEST" if is_best else ""} | {elapsed:.0f}s')
        save_checkpoint(model,optimizer,scheduler,key,epoch,best_auc,history,force=is_best or _exit_flag[0])
        if _exit_flag[0]: print('  [EXIT] stopping — checkpoint pushed'); break
    print(f'Training done | best_val_auc={best_auc:.4f}')
    # Return model,optimizer,scheduler so caller can free all three together
    return model,optimizer,scheduler,history

print('Loops ready')


In [ ]:
# ── 10. Run all models + test evaluation ─────────────────────────────────────
#
# KEY FIX: Between models we:
#   1. free_model(model, optimizer, scheduler) — moves to CPU, deletes, empties cache
#   2. destroy_loaders(train_dl, val_dl, test_dl) — kills worker processes + pinned memory
#   3. gc.collect() + cuda.empty_cache() — second pass
#   4. rebuild loaders fresh for next model
#
# sd_cascade AUROC=0.013 note: inverted detection (model learned backwards for that
# generator). This is a dataset finding, not a bug — keep the result as-is.

test_results={}; all_histories={}

for key in MODEL_SPECS:
    if not MODEL_SPECS[key]['run']: print(f'Skip {key}'); continue

    # ── Build fresh loaders for this model (clean worker processes) ──
    train_dl,val_dl,test_dl=make_loaders()

    model,optimizer,scheduler,history=train_model(key,train_dl,val_dl)
    all_histories[key]=history

    # ── Load best checkpoint for test eval ──
    try:
        dl=hf_hub_download(REPO_ID,ckpt_repo(key),repo_type='dataset',token=TOKEN)
        ckpt=torch.load(dl,map_location=DEVICE,weights_only=False)
        m=model.module if isinstance(model,torch.nn.DataParallel) else model
        m.load_state_dict({k.replace('module.',''):v for k,v in ckpt['model_state'].items()},strict=False)
        print(f'  best ckpt ep={ckpt["epoch"]} auc={ckpt["best_val_auc"]:.4f}')
    except Exception as e: print(f'  WARNING: using last weights ({e.__class__.__name__})')

    print('  Evaluating test set...')
    metrics=evaluate(model,test_dl); test_results[key]=metrics
    o=metrics['overall']
    print(f'  TEST auroc={o["auroc"]:.4f} f1={o["f1"]:.4f} acc={o["accuracy"]:.4f}')
    for gen,gm in metrics.items():
        if gen=='overall': continue
        flag=' ⚠ INVERTED' if gm.get('auroc') is not None and gm['auroc']<0.5 else ''
        print(f'    {gen:<20} auroc={gm["auroc"]:.4f} f1={gm["f1"]:.4f} n={gm["n"]}{flag}')

    # Push partial results so far to HF after each model completes
    partial={'completed_models':list(test_results.keys()),'test_results':test_results}
    push_json(partial,f'{RESULTS_PREFIX}/partial_results.json',f'NB13: {key} complete')

    # ── Full teardown before next model ──
    free_model(model, optimizer, scheduler)
    destroy_loaders(train_dl, val_dl, test_dl)
    del train_dl, val_dl, test_dl
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
    print(f'  Teardown complete | GPU: {torch.cuda.memory_allocated()/1e6:.0f} MB allocated')

print('\nAll models done.')


In [ ]:
# ── 11. Cross-generator generalization (ResNet-50, leave-one-out) ─────────────
CROSS_EPOCHS=5; cross_gen_results={}

# Load cross-gen results from HF if already partially done
try:
    dl=hf_hub_download(REPO_ID,f'{RESULTS_PREFIX}/cross_gen_progress.json',repo_type='dataset',token=TOKEN)
    cross_gen_results=json.load(open(dl))
    print(f'Resumed cross-gen: already done {list(cross_gen_results.keys())}')
except: print('No cross-gen progress found — starting fresh')

if MODEL_SPECS['resnet50']['run']:
    with h5py.File(COMPACT_H5,'r') as f:
        cg_gens=[x.decode() for x in f['generator'][:]]
        cg_splits=[x.decode() for x in f['split'][:]]

    for held_out in GENERATORS:
        if held_out in cross_gen_results:
            print(f'  {held_out}: already done (auroc={cross_gen_results[held_out]["auroc"]:.4f}) — skipping')
            continue
        print(f'\nCross-gen held_out={held_out}')
        tr_idx=[i for i,(g,s) in enumerate(zip(cg_gens,cg_splits)) if s=='train' and (g=='real' or g!=held_out)]
        te_idx=[i for i,(g,s) in enumerate(zip(cg_gens,cg_splits)) if s=='test'  and (g=='real' or g==held_out)]
        ctr_dl,_,cte_dl=make_loaders(train_idx=tr_idx,test_idx=te_idx)
        print(f'  train={len(tr_idx)} test={len(te_idx)}')

        model=build_model('resnet50')
        criterion=nn.CrossEntropyLoss(label_smoothing=0.05)
        optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
        scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=CROSS_EPOCHS,eta_min=LR*0.01)

        for ep in range(CROSS_EPOCHS):
            tr=train_epoch(model,ctr_dl,optimizer,criterion,ep); scheduler.step()
            print(f'  ep{ep} loss={tr["loss"]:.4f} acc={tr["accuracy"]:.4f}')

        m=evaluate(model,cte_dl)['overall']
        cross_gen_results[held_out]={'auroc':m['auroc'],'f1':m['f1'],'accuracy':m['accuracy'],'n':len(te_idx)}
        print(f'  → auroc={m["auroc"]:.4f} f1={m["f1"]:.4f}')

        # Push after each held-out generator completes
        push_json(cross_gen_results,f'{RESULTS_PREFIX}/cross_gen_progress.json',
                  f'cross-gen: {held_out} done')

        free_model(model, optimizer, scheduler)
        destroy_loaders(ctr_dl, cte_dl)
        del ctr_dl, cte_dl
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    print('\nCross-gen summary:')
    for gen,m in cross_gen_results.items(): print(f'  {gen:<20} auroc={m["auroc"]:.4f} f1={m["f1"]:.4f}')
else: print('Skipping cross-gen (resnet50 disabled)')


In [ ]:
# ── 12. Final report → push to HF ─────────────────────────────────────────────
final_report={
    'created_utc':C.now_utc(),'master_seed':MASTER_SEED,
    'hyperparams':{'epochs':EPOCHS,'batch_size':BATCH_SIZE,'lr':LR,'img_size':IMG_SIZE,
                   'label_smoothing':0.05,'optimizer':'AdamW','scheduler':'CosineAnnealingLR'},
    'models':{k:{'description':MODEL_SPECS[k]['description'],'timm_name':MODEL_SPECS[k]['timm_name'],
                 'test_metrics':test_results.get(k,{})} for k in MODEL_SPECS if MODEL_SPECS[k]['run']},
    'cross_generator_generalization':cross_gen_results,
    'sniff_test_reference':{'description':'ResNet-18 1-epoch probe (validation_report.json)',
        'results':{'sd15':0.853,'sdxl':0.973,'flux_schnell':0.877,'kandinsky22':0.960,
                   'pixart_sigma':0.932,'sd_cascade':1.0,'__overall__':0.847}}
}
print('\n'+'='*65+'\nFINAL RESULTS — TEST SET\n'+'='*65)
print(f'{"Model":<22} {"AUROC":>8} {"F1":>8} {"Accuracy":>10}')
for k,d in final_report['models'].items():
    m=d['test_metrics'].get('overall',{})
    if m: print(f'{k:<22} {m["auroc"]:>8.4f} {m["f1"]:>8.4f} {m["accuracy"]:>10.4f}')
if cross_gen_results:
    print('\nCROSS-GEN (ResNet-50)'); print(f'{"Held-out":<22} {"AUROC":>8} {"F1":>8}')
    for gen,m in cross_gen_results.items(): print(f'{gen:<22} {m["auroc"]:>8.4f} {m["f1"]:>8.4f}')
push_json(final_report,f'{RESULTS_PREFIX}/baseline_results.json','NB13: COMPLETE')
print('\nbaseline_results.json pushed. NB13 complete.')

In [ ]:
# ── 13. LaTeX tables (copy-paste into paper) ──────────────────────────────────
MD={'resnet50':'ResNet-50','efficientnet_b4':'EfficientNet-B4','vit_b16':'ViT-B/16'}
GD={'sd15':'SD~1.5','sdxl':'SDXL','flux_schnell':'FLUX.1-schnell','kandinsky22':'Kandinsky~2.2',
    'pixart_sigma':'PixArt-$\\Sigma$','sd_cascade':'SD~Cascade'}
sniff=final_report['sniff_test_reference']['results']

print('% TABLE 1 — Sniff test')
print(r'\begin{table}[t]\centering\caption{ResNet-18 one-epoch detectability probe.}\label{tab:sniff}')
print(r'\begin{tabular}{lc}\toprule Generator & Accuracy\\\midrule')
for gk,gd in GD.items():
    v=sniff.get(gk,0); flag=r'$^\dagger$' if v>0.95 else ''
    print(f'{gd} & {v:.3f}{flag} \\\\')
print(f'Overall & {sniff["__overall__"]:.3f} \\\\')
print(r'\bottomrule\end{tabular}{\scriptsize $^\dagger$~above 0.95 threshold}\end{table}')
print()

print('% TABLE 2 — Overall baselines')
print(r'\begin{table}[t]\centering\caption{Baseline detector performance on test split.}\label{tab:baselines}')
print(r'\begin{tabular}{lccc}\toprule Model & AUROC & F1 & Acc.\\\midrule')
for k in MODEL_SPECS:
    if not MODEL_SPECS[k]['run']: continue
    m=test_results.get(k,{}).get('overall',{})
    row=f'{m["auroc"]:.4f} & {m["f1"]:.4f} & {m["accuracy"]:.4f}' if m else '--&--&--'
    print(f'{MD.get(k,k)} & {row} \\\\')
print(r'\bottomrule\end{tabular}\end{table}')
print()

best_k=max((k for k in test_results if test_results[k].get('overall')),
           key=lambda k:test_results[k]['overall'].get('auroc',0),default=None)
if best_k:
    print(f'% TABLE 3 — Per-generator ({best_k})')
    print(r'\begin{table}[t]\centering\caption{Per-generator detection results.}\label{tab:per_gen}')
    print(r'\begin{tabular}{lccc}\toprule Generator & AUROC & F1 & Acc.\\\midrule')
    for gk,gd in GD.items():
        m=test_results[best_k].get(gk,{})
        row=f'{m["auroc"]:.4f} & {m["f1"]:.4f} & {m["accuracy"]:.4f}' if m else '--&--&--'
        print(f'{gd} & {row} \\\\')
    ov=test_results[best_k].get('overall',{})
    if ov: print(r'\midrule'); print(f'\\textbf{{Overall}} & \\textbf{{{ov["auroc"]:.4f}}} & \\textbf{{{ov["f1"]:.4f}}} & \\textbf{{{ov["accuracy"]:.4f}}} \\\\')
    print(r'\bottomrule\end{tabular}\end{table}')
    print()

if cross_gen_results:
    print('% TABLE 4 — Cross-generator generalization')
    print(r'\begin{table}[t]\centering\caption{Cross-generator generalization (ResNet-50, 5 epochs).}\label{tab:cross_gen}')
    print(r'\begin{tabular}{lcc}\toprule Held-out generator & AUROC & F1\\\midrule')
    for gk,m in cross_gen_results.items(): print(f'{GD.get(gk,gk)} & {m["auroc"]:.4f} & {m["f1"]:.4f} \\\\')
    print(r'\bottomrule\end{tabular}\end{table}')
print('\n% All 4 tables generated.')

## NB13 complete

**Peak memory:** ~630 MB disk + ~3 GB RAM + ~1.5 GB GPU. No OOM possible.

**HF after run:**
```
results/baseline_experiments/
  compact.h5                    ← resume key for Phase 1
  nb13_extract_progress.json
  baseline_results.json         ← ALL metrics
  resnet50_progress.json
  efficientnet_b4_progress.json
  vit_b16_progress.json
  checkpoints/resnet50_best.pt
  checkpoints/efficientnet_b4_best.pt
  checkpoints/vit_b16_best.pt
```

**On any restart:** re-run all cells top to bottom. Phase 1 resumes from HF. Training resumes from last best checkpoint.